# 👾 PixelCNN using Tensorflow distributions

In this notebook, we'll walk through the steps required to train your own PixelCNN on the fashion MNIST dataset using Tensorflow distributions

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torchvision import datasets
from transformers import Trainer, TrainingArguments, TrainerCallback

# PyTorch has no PixelCNN distribution like TensorFlow Probability, so we use
# a compact PixelCNN++ implementation that lives next to this notebook
from pixelcnn_pp_pytorch import PixelCNN

from notebooks.utils_pytorch import display

## 0. Parameters <a name="parameters"></a>

In [ ]:
IMAGE_SIZE = 32
N_COMPONENTS = 5
EPOCHS = 10
BATCH_SIZE = 128

## 1. Prepare the data <a name="prepare"></a>

In [ ]:
# Load the data
train_set = datasets.FashionMNIST(root="../../../data", train=True, download=True)
x_train = train_set.data.numpy()  # uint8 array of shape (60000, 28, 28)

In [ ]:
# Preprocess the data


def preprocess(imgs):
    # (N, 28, 28) -> (N, 1, 28, 28): PyTorch images are channels-first (NCHW)
    imgs = torch.tensor(imgs, dtype=torch.float32).unsqueeze(1)
    imgs = F.interpolate(
        imgs, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
    )
    # Keep pixel values in [0, 255], rounded to whole numbers because the
    # PixelCNN models discrete pixel values
    return imgs.round()


input_data = preprocess(x_train)

In [ ]:
# Show some items of clothing from the training set
display(input_data)

## 2. Build the PixelCNN <a name="build"></a>

In [ ]:
# Define a Pixel CNN network (see pixelcnn_pp_pytorch.py). Like the TFP version,
# it is a distribution over images with log_prob() and sample() methods
dist = PixelCNN(
    image_shape=(1, IMAGE_SIZE, IMAGE_SIZE),  # channels first
    num_resnet=1,
    num_hierarchies=2,
    num_filters=32,
    num_logistic_mix=N_COMPONENTS,
    dropout_p=0.3,
)


# The Trainer needs a model whose forward() returns the loss: here the
# negative log-likelihood of the images (like add_loss in the Keras version)
class PixelCNNModel(nn.Module):
    def __init__(self, dist):
        super().__init__()
        self.dist = dist

    def forward(self, pixel_values):
        log_prob = self.dist.log_prob(pixel_values)
        return {"loss": -log_prob.mean(), "log_prob": log_prob}


pixelcnn = PixelCNNModel(dist)
print(pixelcnn)
print(f"Total params: {sum(p.numel() for p in pixelcnn.parameters()):,}")

## 3. Train the PixelCNN <a name="train"></a>

In [ ]:
# TrainingArguments holds the optimizer and training settings
# (the Trainer uses AdamW; with weight_decay=0 it is plain Adam)
training_args = TrainingArguments(
    output_dir="./checkpoint",
    learning_rate=0.001,
    lr_scheduler_type="constant",  # Keras keeps the learning rate fixed
    max_grad_norm=0.0,  # the Trainer clips gradients by default; Keras doesn't
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    logging_strategy="epoch",
    save_strategy="no",
    report_to="tensorboard",
    remove_unused_columns=False,  # our dataset keys are already what the model needs
)

In [ ]:
# TensorBoard logs go to ./logs (set with an environment variable in transformers 5)
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"


# Samples images; the Trainer calls on_epoch_end after each epoch
class ImageGenerator(TrainerCallback):
    def __init__(self, num_img):
        self.num_img = num_img

    def generate(self):
        return dist.sample(self.num_img).cpu().numpy()

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = round(state.epoch) - 1  # state.epoch counts completed epochs
        generated_images = self.generate()
        display(
            generated_images,
            n=self.num_img,
            save_to="./output/generated_img_%03d.png" % (epoch),
        )


img_generator_callback = ImageGenerator(num_img=2)

In [ ]:
# The Trainer feeds each example to the model as keyword arguments,
# so every example is a dict whose keys match model.forward()
class ImageDataset(Dataset):
    def __init__(self, images):
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return {"pixel_values": self.images[idx]}


trainer = Trainer(
    model=pixelcnn,
    args=training_args,
    train_dataset=ImageDataset(input_data),
    callbacks=[img_generator_callback],
)
trainer.train()

## 4. Generate images <a name="generate"></a>

In [ ]:
generated_images = img_generator_callback.generate()

In [ ]:
display(generated_images, n=img_generator_callback.num_img)